# Cleanup Notebook
# This notebook reproduces the logic from CleanupFile.py as interactive steps (load, inspect, clean, save).
# Use the cells below in order. Each step prints diagnostics before making destructive changes.


## Overview

This notebook converts the `CleanupFile.py` script into interactive steps. It is designed so you can run each cell, inspect diagnostic output, tweak thresholds, and re-run. The notebook covers:

- Loading files from `Extracted Data`
- Inspecting shapes / columns / dtypes
- Removing duplicates
- Handling missing / infinite values
- Converting datatypes (timestamps, numeric conversion)
- Filtering broadcast/multicast/empty-packet rows (safe diagnostics before dropping)
- Normalizing IPs/ports/protocols
- Removing outliers using an IQR approach (with reporting)
- Saving cleaned files to `Cleaned Data`

Run cells top to bottom. Each cleaning step prints counts before/after and will not proceed silently if a step would remove all rows.


In [1]:
# Cell: Imports and settings
import os
from pathlib import Path
import re
import glob
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# pandas display options for interactive use
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)


In [2]:
# Cell: Project paths (detect project root containing 'Extracted Data')
from pathlib import Path

cwd = Path.cwd()
project_root = cwd
while not (project_root / 'Extracted Data').exists() and project_root != project_root.parent:
    project_root = project_root.parent

if not (project_root / 'Extracted Data').exists():
    raise FileNotFoundError("Could not find 'Extracted Data' folder - please run the notebook from the repo or set project_root manually.")

input_dir = project_root / 'Extracted Data'
output_dir = project_root / 'Cleaned Data'
output_dir.mkdir(exist_ok=True)

print('Project root:', project_root)
print('Input dir:', input_dir)
print('Output dir:', output_dir)

# List CSV files
csv_files = sorted([p for p in input_dir.glob('*.csv')])
print('\nFound CSV files:')
for p in csv_files:
    print(' -', p.name)


Project root: /Users/divyanshioberoi/Desktop/IIT/Fall 2025/CS597/RealTime-Network-Traffic-Classifier
Input dir: /Users/divyanshioberoi/Desktop/IIT/Fall 2025/CS597/RealTime-Network-Traffic-Classifier/Extracted Data
Output dir: /Users/divyanshioberoi/Desktop/IIT/Fall 2025/CS597/RealTime-Network-Traffic-Classifier/Cleaned Data

Found CSV files:
 - Friday-WorkingHours.pcap_Flow.csv
 - Monday-WorkingHours.pcap_Flow.csv
 - Thursday-WorkingHours.pcap_Flow.csv
 - Tuesday-WorkingHours.pcap_Flow.csv
 - Wednesday-workingHours.pcap_Flow.csv


In [3]:
# Cell: Helper functions
import ipaddress

def safe_read_csv(path, nrows=None):
    try:
        return pd.read_csv(path, low_memory=False, nrows=nrows)
    except Exception as e:
        print(f'Error reading {path}:', e)
        raise


def save_csv(df, out_path):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_path, index=False)
    print(f"Saved: {out_path}  (shape={df.shape})")


def show_schema(df, n=5):
    print('Shape:', df.shape)
    print('\nDtypes:')
    print(df.dtypes)
    print('\nNull counts:')
    print(df.isnull().sum().sort_values(ascending=False).head(10))
    display(df.head(n))


def is_valid_ipv4(s):
    try:
        ipaddress.IPv4Address(s)
        return True
    except Exception:
        return False


In [4]:
# Step 1: Load one CSV and preview
if not csv_files:
    raise FileNotFoundError('No CSV files found in input dir')

sample_file = csv_files[0]
print('Loading sample file:', sample_file.name)
df = safe_read_csv(sample_file, nrows=None)
show_schema(df, n=3)


Loading sample file: Friday-WorkingHours.pcap_Flow.csv
Shape: (703283, 84)

Dtypes:
Flow ID       object
Src IP        object
Src Port       int64
Dst IP        object
Dst Port       int64
              ...   
Idle Mean    float64
Idle Std     float64
Idle Max     float64
Idle Min     float64
Label         object
Length: 84, dtype: object

Null counts:
Flow Byts/s         480
URG Flag Cnt          0
Fwd Pkts/b Avg        0
Fwd Byts/b Avg        0
Bwd Seg Size Avg      0
Fwd Seg Size Avg      0
Pkt Size Avg          0
Down/Up Ratio         0
ECE Flag Cnt          0
CWE Flag Count        0
dtype: int64


,Flow ID,Src IP,Src Port,Dst IP,Dst Port,Protocol,Timestamp,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,TotLen Fwd Pkts,TotLen Bwd Pkts,Fwd Pkt Len Max,Fwd Pkt Len Min,Fwd Pkt Len Mean,Fwd Pkt Len Std,Bwd Pkt Len Max,Bwd Pkt Len Min,Bwd Pkt Len Mean,Bwd Pkt Len Std,Flow Byts/s,Flow Pkts/s,Flow IAT Mean,Flow IAT Std,Flow IAT Max,Flow IAT Min,Fwd IAT Tot,Fwd IAT Mean,Fwd IAT Std,Fwd IAT Max,Fwd IAT Min,Bwd IAT Tot,Bwd IAT Mean,Bwd IAT Std,Bwd IAT Max,Bwd IAT Min,Fwd PSH Flags,Bwd PSH Flags,Fwd URG Flags,Bwd URG Flags,Fwd Header Len,Bwd Header Len,Fwd Pkts/s,Bwd Pkts/s,Pkt Len Min,Pkt Len Max,Pkt Len Mean,Pkt Len Std,Pkt Len Var,FIN Flag Cnt,SYN Flag Cnt,RST Flag Cnt,PSH Flag Cnt,ACK Flag Cnt,URG Flag Cnt,CWE Flag Count,ECE Flag Cnt,Down/Up Ratio,Pkt Size Avg,Fwd Seg Size Avg,Bwd Seg Size Avg,Fwd Byts/b Avg,Fwd Pkts/b Avg,Fwd Blk Rate Avg,Bwd Byts/b Avg,Bwd Pkts/b Avg,Bwd Blk Rate Avg,Subflow Fwd Pkts,Subflow Fwd Byts,Subflow Bwd Pkts,Subflow Bwd Byts,Init Fwd Win Byts,Init Bwd Win Byts,Fwd Act Data Pkts,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,192.168.10.3-192.168.10.9-88-1031-6,192.168.10.9,1031,192.168.10.3,88,6,07/07/2017 07:00:35 AM,617,6,5,466.0,414.0,233.0,0.0,77.666667,120.320683,207.0,0.0,82.8,113.378569,1.426256e+06,17828.200972,61.700000,132.036232,430.0,1.0,570.0,114.000000,225.657927,516.0,1.0,535.0,133.75,214.206092,451.0,4.0,0,0,0,0,132,136,9724.473258,8103.727715,0.0,233.0,73.333333,108.603812,11794.787879,0,1,0,0,0,0,0,0,0.0,80.000000,77.666667,82.8,0,0,0,0,0,0,6,466,5,414,-1,2053,2,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No Label
1,192.168.10.3-192.168.10.9-88-1031-6,192.168.10.3,88,192.168.10.9,1031,6,07/07/2017 07:00:35 AM,8,0,2,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.000000e+00,250000.000000,8.000000,0.000000,8.0,8.0,0.0,0.000000,0.000000,0.0,0.0,8.0,8.00,0.000000,8.0,8.0,0,0,0,0,0,40,0.000000,250000.000000,0.0,0.0,0.000000,0.000000,0.000000,0,0,0,0,1,0,0,0,0.0,0.000000,0.000000,0.0,0,0,0,0,0,0,0,0,2,0,-1,255,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No Label
2,192.168.10.3-192.168.10.9-88-1032-6,192.168.10.9,1032,192.168.10.3,88,6,07/07/2017 07:00:35 AM,881,8,5,626.0,3064.0,313.0,0.0,78.250000,144.890846,1532.0,0.0,612.8,839.110958,4.188422e+06,14755.959137,73.416667,193.311552,683.0,1.0,878.0,125.428571,280.126672,758.0,1.0,806.0,201.50,370.067111,756.0,4.0,0,0,0,0,172,136,9080.590238,5675.368899,0.0,1532.0,263.571429,548.943561,301339.032967,0,1,0,0,0,0,0,0,0.0,283.846154,78.250000,612.8,0,0,0,0,0,0,8,626,5,3064,-1,2053,2,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No Label


In [5]:
# Step 2: Remove duplicates (diagnostic + apply)
print('Rows before dedupe:', len(df))
dups = df.duplicated().sum()
print('Duplicate rows detected:', dups)
if dups>0:
    df = df.drop_duplicates()
    print('Rows after dedupe:', len(df))
else:
    print('No duplicates removed')


Rows before dedupe: 703283
Duplicate rows detected: 5
Rows after dedupe: 703278


In [6]:
# Step 3: Handle missing & infinite values
print('\nMissing values before:')
print(df.isna().sum().sort_values(ascending=False).head(10))

# Replace +inf/-inf with NaN
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Numeric columns
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
for col in numeric_cols:
    if df[col].isnull().any():
        if any(x in col.lower() for x in ['packet', 'pkt', 'byte', 'len']):
            df[col] = df[col].fillna(0)
        else:
            df[col] = df[col].fillna(df[col].median())

# Categorical/object columns
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
for col in cat_cols:
    if df[col].isnull().any():
        try:
            mode = df[col].mode()[0]
            df[col] = df[col].fillna(mode)
        except Exception:
            df[col] = df[col].fillna('')

print('\nMissing values after:')
print(df.isna().sum().sort_values(ascending=False).head(10))



Missing values before:
Flow Byts/s         480
URG Flag Cnt          0
Fwd Pkts/b Avg        0
Fwd Byts/b Avg        0
Bwd Seg Size Avg      0
Fwd Seg Size Avg      0
Pkt Size Avg          0
Down/Up Ratio         0
ECE Flag Cnt          0
CWE Flag Count        0
dtype: int64

Missing values after:
Flow ID             0
ACK Flag Cnt        0
Fwd Byts/b Avg      0
Bwd Seg Size Avg    0
Fwd Seg Size Avg    0
Pkt Size Avg        0
Down/Up Ratio       0
ECE Flag Cnt        0
CWE Flag Count      0
URG Flag Cnt        0
dtype: int64


In [7]:
# Step 4: Convert datatypes and extract time features
# Find timestamp-like columns (names containing 'time' or 'timestamp')
timestamp_cols = [c for c in df.columns if 'time' in c.lower() or 'timestamp' in c.lower()]
print('Timestamp-like columns detected:', timestamp_cols)

for col in timestamp_cols:
    try:
        df[col] = pd.to_datetime(df[col])
        df[f'{col}_dayofweek'] = df[col].dt.day_name()
        df[f'{col}_dayofweek_int'] = df[col].dt.dayofweek  # Monday=0
        df[f'{col}_date'] = df[col].dt.date
        print(f'  Parsed {col} -> datetime, added {col}_dayofweek, {col}_dayofweek_int, {col}_date')
    except Exception as e:
        print('  Could not parse', col, e)

# Convert other numeric-like columns (attempt coercion)
for col in df.select_dtypes(include=['object']).columns:
    # skip ip-like columns
    if 'ip' in col.lower() or 'protocol' in col.lower():
        continue
    coerced = pd.to_numeric(df[col], errors='coerce')
    # If coercion produces many non-nulls, keep it
    if coerced.notna().sum() >= 0.9 * len(coerced):
        df[col] = coerced
        print('Converted to numeric:', col)

show_schema(df, n=3)


Timestamp-like columns detected: ['Timestamp']
  Parsed Timestamp -> datetime, added Timestamp_dayofweek, Timestamp_dayofweek_int, Timestamp_date
Shape: (703278, 87)

Dtypes:
Flow ID                     object
Src IP                      object
Src Port                     int64
Dst IP                      object
Dst Port                     int64
                            ...   
Idle Min                   float64
Label                       object
Timestamp_dayofweek         object
Timestamp_dayofweek_int      int32
Timestamp_date              object
Length: 87, dtype: object

Null counts:
Flow ID             0
CWE Flag Count      0
Fwd Blk Rate Avg    0
Fwd Pkts/b Avg      0
Fwd Byts/b Avg      0
Bwd Seg Size Avg    0
Fwd Seg Size Avg    0
Pkt Size Avg        0
Down/Up Ratio       0
ECE Flag Cnt        0
dtype: int64


,Flow ID,Src IP,Src Port,Dst IP,Dst Port,Protocol,Timestamp,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,TotLen Fwd Pkts,TotLen Bwd Pkts,Fwd Pkt Len Max,Fwd Pkt Len Min,Fwd Pkt Len Mean,Fwd Pkt Len Std,Bwd Pkt Len Max,Bwd Pkt Len Min,Bwd Pkt Len Mean,Bwd Pkt Len Std,Flow Byts/s,Flow Pkts/s,Flow IAT Mean,Flow IAT Std,Flow IAT Max,Flow IAT Min,Fwd IAT Tot,Fwd IAT Mean,Fwd IAT Std,Fwd IAT Max,Fwd IAT Min,Bwd IAT Tot,Bwd IAT Mean,Bwd IAT Std,Bwd IAT Max,Bwd IAT Min,Fwd PSH Flags,Bwd PSH Flags,Fwd URG Flags,Bwd URG Flags,Fwd Header Len,Bwd Header Len,Fwd Pkts/s,Bwd Pkts/s,Pkt Len Min,Pkt Len Max,Pkt Len Mean,Pkt Len Std,Pkt Len Var,FIN Flag Cnt,SYN Flag Cnt,RST Flag Cnt,PSH Flag Cnt,ACK Flag Cnt,URG Flag Cnt,CWE Flag Count,ECE Flag Cnt,Down/Up Ratio,Pkt Size Avg,Fwd Seg Size Avg,Bwd Seg Size Avg,Fwd Byts/b Avg,Fwd Pkts/b Avg,Fwd Blk Rate Avg,Bwd Byts/b Avg,Bwd Pkts/b Avg,Bwd Blk Rate Avg,Subflow Fwd Pkts,Subflow Fwd Byts,Subflow Bwd Pkts,Subflow Bwd Byts,Init Fwd Win Byts,Init Bwd Win Byts,Fwd Act Data Pkts,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label,Timestamp_dayofweek,Timestamp_dayofweek_int,Timestamp_date
0,192.168.10.3-192.168.10.9-88-1031-6,192.168.10.9,1031,192.168.10.3,88,6,2017-07-07 07:00:35,617,6,5,466.0,414.0,233.0,0.0,77.666667,120.320683,207.0,0.0,82.8,113.378569,1.426256e+06,17828.200972,61.700000,132.036232,430.0,1.0,570.0,114.000000,225.657927,516.0,1.0,535.0,133.75,214.206092,451.0,4.0,0,0,0,0,132,136,9724.473258,8103.727715,0.0,233.0,73.333333,108.603812,11794.787879,0,1,0,0,0,0,0,0,0.0,80.000000,77.666667,82.8,0,0,0,0,0,0,6,466,5,414,-1,2053,2,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No Label,Friday,4,2017-07-07
1,192.168.10.3-192.168.10.9-88-1031-6,192.168.10.3,88,192.168.10.9,1031,6,2017-07-07 07:00:35,8,0,2,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.000000e+00,250000.000000,8.000000,0.000000,8.0,8.0,0.0,0.000000,0.000000,0.0,0.0,8.0,8.00,0.000000,8.0,8.0,0,0,0,0,0,40,0.000000,250000.000000,0.0,0.0,0.000000,0.000000,0.000000,0,0,0,0,1,0,0,0,0.0,0.000000,0.000000,0.0,0,0,0,0,0,0,0,0,2,0,-1,255,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No Label,Friday,4,2017-07-07
2,192.168.10.3-192.168.10.9-88-1032-6,192.168.10.9,1032,192.168.10.3,88,6,2017-07-07 07:00:35,881,8,5,626.0,3064.0,313.0,0.0,78.250000,144.890846,1532.0,0.0,612.8,839.110958,4.188422e+06,14755.959137,73.416667,193.311552,683.0,1.0,878.0,125.428571,280.126672,758.0,1.0,806.0,201.50,370.067111,756.0,4.0,0,0,0,0,172,136,9080.590238,5675.368899,0.0,1532.0,263.571429,548.943561,301339.032967,0,1,0,0,0,0,0,0,0.0,283.846154,78.250000,612.8,0,0,0,0,0,0,8,626,5,3064,-1,2053,2,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No Label,Friday,4,2017-07-07


In [8]:
# Step 5: Filter irrelevant traffic (broadcast / multicast / zero-packet) - diagnostics first
# Candidate columns for packet size/length
pkt_cols = [c for c in df.columns if any(k in c.lower() for k in ['packet','pkt','len','size','bytes','byts'])]
print('Candidate packet/length columns:', pkt_cols)

# Build masks safely
masks = {}
# Broadcast
for ip_col in [c for c in df.columns if 'dst' in c.lower() and 'ip' in c.lower()]:
    dst_s = df[ip_col].fillna('').astype(str).str.strip()
    masks[f'broadcast_{ip_col}'] = dst_s == '255.255.255.255'
    print(ip_col, 'broadcast count ->', masks[f'broadcast_{ip_col}'].sum())

# Multicast: first octet between 224 and 239
for ip_col in [c for c in df.columns if 'dst' in c.lower() and 'ip' in c.lower()]:
    dst_s = df[ip_col].fillna('').astype(str).str.strip()
    first_oct = dst_s.str.split('.', expand=True)[0]
    mask_multicast = first_oct.map(lambda x: str(x).isdigit() and 224 <= int(x) <= 239)
    masks[f'multicast_{ip_col}'] = mask_multicast
    print(ip_col, 'multicast count ->', mask_multicast.sum())

# Zero or NaN packet size
mask_zero_pkt = pd.Series(False, index=df.index)
for c in pkt_cols:
    pkt_num = pd.to_numeric(df[c], errors='coerce')
    mask_zero_pkt = mask_zero_pkt | pkt_num.isna() | (pkt_num == 0)
print('Zero-or-NaN packet rows (union across pkt cols):', mask_zero_pkt.sum())

# Combine masks (union)
combined_mask = pd.Series(False, index=df.index)
for k,v in masks.items():
    combined_mask = combined_mask | v
combined_mask = combined_mask | mask_zero_pkt

print('\nCombined rows to remove:', combined_mask.sum(), 'out of', len(df))

# Guard: do not drop if it removes all rows
if combined_mask.sum() >= len(df):
    print('WARNING: combined filter would remove all rows; not applying. Inspect masks and column names.')
else:
    before = len(df)
    df = df.loc[~combined_mask].copy()
    print('Applied filter: rows', before, '->', len(df))


Candidate packet/length columns: ['Tot Fwd Pkts', 'Tot Bwd Pkts', 'TotLen Fwd Pkts', 'TotLen Bwd Pkts', 'Fwd Pkt Len Max', 'Fwd Pkt Len Min', 'Fwd Pkt Len Mean', 'Fwd Pkt Len Std', 'Bwd Pkt Len Max', 'Bwd Pkt Len Min', 'Bwd Pkt Len Mean', 'Bwd Pkt Len Std', 'Flow Byts/s', 'Flow Pkts/s', 'Fwd Header Len', 'Bwd Header Len', 'Fwd Pkts/s', 'Bwd Pkts/s', 'Pkt Len Min', 'Pkt Len Max', 'Pkt Len Mean', 'Pkt Len Std', 'Pkt Len Var', 'Pkt Size Avg', 'Fwd Seg Size Avg', 'Bwd Seg Size Avg', 'Fwd Byts/b Avg', 'Fwd Pkts/b Avg', 'Bwd Byts/b Avg', 'Bwd Pkts/b Avg', 'Subflow Fwd Pkts', 'Subflow Fwd Byts', 'Subflow Bwd Pkts', 'Subflow Bwd Byts', 'Init Fwd Win Byts', 'Init Bwd Win Byts', 'Fwd Act Data Pkts', 'Fwd Seg Size Min']
Dst IP broadcast count -> 78
Dst IP multicast count -> 236
Zero-or-NaN packet rows (union across pkt cols): 703278

Combined rows to remove: 703278 out of 703278


In [10]:
# Step 6: Normalize IPs/ports/protocols (robust to non-string dtypes)
# Normalize column names to lowercase and strip
orig_cols = df.columns.tolist()
df.columns = [c.lower().strip() for c in df.columns]
print('Columns normalized (lowercased)')

# Trim string-like columns (object/string dtype)
for col in df.select_dtypes(include=['object', 'string']).columns:
    # convert to pandas StringDtype for safe .str operations while preserving NA
    df[col] = df[col].astype('string').str.strip()

# Protocol normalization: convert to pandas string dtype then lowercase/strip
proto_cols = [c for c in df.columns if 'protocol' in c]
for c in proto_cols:
    # convert to StringDtype first to avoid AttributeError
    df[c] = df[c].astype('string').str.lower().str.strip()
    # If you prefer to fill missing protocol values with a placeholder, uncomment:
    # df[c] = df[c].fillna('unknown')

# Ports to numeric (Int64) and validate
port_cols = [c for c in df.columns if 'port' in c]
for c in port_cols:
    df[c] = pd.to_numeric(df[c], errors='coerce').astype('Int64')
    invalid = df[c].isna().sum()
    print(f'Port column {c} invalid/coerced-to-NA count:', invalid)
    # do NOT automatically drop rows here; leave decision to user or next step

show_schema(df, n=2)


Columns normalized (lowercased)
Port column src port invalid/coerced-to-NA count: 0
Port column dst port invalid/coerced-to-NA count: 0
Shape: (703278, 87)

Dtypes:
flow id                    string[python]
src ip                     string[python]
src port                            Int64
dst ip                     string[python]
dst port                            Int64
                                ...      
idle min                          float64
label                      string[python]
timestamp_dayofweek        string[python]
timestamp_dayofweek_int             int32
timestamp_date             string[python]
Length: 87, dtype: object

Null counts:
flow id             0
cwe flag count      0
fwd blk rate avg    0
fwd pkts/b avg      0
fwd byts/b avg      0
bwd seg size avg    0
fwd seg size avg    0
pkt size avg        0
down/up ratio       0
ece flag cnt        0
dtype: int64


,flow id,src ip,src port,dst ip,dst port,protocol,timestamp,flow duration,tot fwd pkts,tot bwd pkts,totlen fwd pkts,totlen bwd pkts,fwd pkt len max,fwd pkt len min,fwd pkt len mean,fwd pkt len std,bwd pkt len max,bwd pkt len min,bwd pkt len mean,bwd pkt len std,flow byts/s,flow pkts/s,flow iat mean,flow iat std,flow iat max,flow iat min,fwd iat tot,fwd iat mean,fwd iat std,fwd iat max,fwd iat min,bwd iat tot,bwd iat mean,bwd iat std,bwd iat max,bwd iat min,fwd psh flags,bwd psh flags,fwd urg flags,bwd urg flags,fwd header len,bwd header len,fwd pkts/s,bwd pkts/s,pkt len min,pkt len max,pkt len mean,pkt len std,pkt len var,fin flag cnt,syn flag cnt,rst flag cnt,psh flag cnt,ack flag cnt,urg flag cnt,cwe flag count,ece flag cnt,down/up ratio,pkt size avg,fwd seg size avg,bwd seg size avg,fwd byts/b avg,fwd pkts/b avg,fwd blk rate avg,bwd byts/b avg,bwd pkts/b avg,bwd blk rate avg,subflow fwd pkts,subflow fwd byts,subflow bwd pkts,subflow bwd byts,init fwd win byts,init bwd win byts,fwd act data pkts,fwd seg size min,active mean,active std,active max,active min,idle mean,idle std,idle max,idle min,label,timestamp_dayofweek,timestamp_dayofweek_int,timestamp_date
0,192.168.10.3-192.168.10.9-88-1031-6,192.168.10.9,1031,192.168.10.3,88,6,2017-07-07 07:00:35,617,6,5,466.0,414.0,233.0,0.0,77.666667,120.320683,207.0,0.0,82.8,113.378569,1.426256e+06,17828.200972,61.7,132.036232,430.0,1.0,570.0,114.0,225.657927,516.0,1.0,535.0,133.75,214.206092,451.0,4.0,0,0,0,0,132,136,9724.473258,8103.727715,0.0,233.0,73.333333,108.603812,11794.787879,0,1,0,0,0,0,0,0,0.0,80.0,77.666667,82.8,0,0,0,0,0,0,6,466,5,414,-1,2053,2,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No Label,Friday,4,2017-07-07
1,192.168.10.3-192.168.10.9-88-1031-6,192.168.10.3,88,192.168.10.9,1031,6,2017-07-07 07:00:35,8,0,2,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.000000e+00,250000.000000,8.0,0.000000,8.0,8.0,0.0,0.0,0.000000,0.0,0.0,8.0,8.00,0.000000,8.0,8.0,0,0,0,0,0,40,0.000000,250000.000000,0.0,0.0,0.000000,0.000000,0.000000,0,0,0,0,1,0,0,0,0.0,0.0,0.000000,0.0,0,0,0,0,0,0,0,0,2,0,-1,255,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No Label,Friday,4,2017-07-07


In [11]:
# Step 7: Remove outliers (IQR-based) - report removals stepwise
numeric_cols = [c for c in df.select_dtypes(include=[np.number]).columns if not any(x in c for x in ['port'])]
print('Numeric columns to check for outliers:', numeric_cols[:20])

total_before = len(df)
for col in numeric_cols:
    try:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        if pd.isna(IQR) or IQR == 0:
            continue
        lower = Q1 - 3 * IQR
        upper = Q3 + 3 * IQR
        outliers = ((df[col] < lower) | (df[col] > upper)).sum()
        if outliers > 0:
            # apply filter
            df = df[(df[col] >= lower) & (df[col] <= upper)]
            print(f'Removed {outliers} rows as outliers from {col} -> new length {len(df)}')
    except Exception as e:
        print('Skipping', col, e)

print('Outlier removal completed: rows', total_before, '->', len(df))


Numeric columns to check for outliers: ['flow duration', 'tot fwd pkts', 'tot bwd pkts', 'totlen fwd pkts', 'totlen bwd pkts', 'fwd pkt len max', 'fwd pkt len min', 'fwd pkt len mean', 'fwd pkt len std', 'bwd pkt len max', 'bwd pkt len min', 'bwd pkt len mean', 'bwd pkt len std', 'flow byts/s', 'flow pkts/s', 'flow iat mean', 'flow iat std', 'flow iat max', 'flow iat min', 'fwd iat tot']
Removed 114337 rows as outliers from flow duration -> new length 588941
Removed 13506 rows as outliers from tot fwd pkts -> new length 575435
Removed 35164 rows as outliers from tot bwd pkts -> new length 540271
Removed 25355 rows as outliers from totlen fwd pkts -> new length 514916
Removed 39300 rows as outliers from totlen bwd pkts -> new length 475616
Removed 10 rows as outliers from fwd pkt len max -> new length 475606
Removed 1164 rows as outliers from bwd pkt len max -> new length 474442
Removed 777 rows as outliers from bwd pkt len min -> new length 473665
Removed 11171 rows as outliers from bw

In [12]:
# Step 8: Save cleaned sample and show summary
sample_out = output_dir / f'cleaned_{sample_file.name}'
print('Saving to', sample_out)
save_csv(df, sample_out)
print('\nFinal schema for saved file:')
show_schema(df, n=2)


Saving to /Users/divyanshioberoi/Desktop/IIT/Fall 2025/CS597/RealTime-Network-Traffic-Classifier/Cleaned Data/cleaned_Friday-WorkingHours.pcap_Flow.csv
Saved: /Users/divyanshioberoi/Desktop/IIT/Fall 2025/CS597/RealTime-Network-Traffic-Classifier/Cleaned Data/cleaned_Friday-WorkingHours.pcap_Flow.csv  (shape=(185713, 87))

Final schema for saved file:
Shape: (185713, 87)

Dtypes:
flow id                    string[python]
src ip                     string[python]
src port                            Int64
dst ip                     string[python]
dst port                            Int64
                                ...      
idle min                          float64
label                      string[python]
timestamp_dayofweek        string[python]
timestamp_dayofweek_int             int32
timestamp_date             string[python]
Length: 87, dtype: object

Null counts:
flow id             0
cwe flag count      0
fwd blk rate avg    0
fwd pkts/b avg      0
fwd byts/b avg      0
bwd se

,flow id,src ip,src port,dst ip,dst port,protocol,timestamp,flow duration,tot fwd pkts,tot bwd pkts,totlen fwd pkts,totlen bwd pkts,fwd pkt len max,fwd pkt len min,fwd pkt len mean,fwd pkt len std,bwd pkt len max,bwd pkt len min,bwd pkt len mean,bwd pkt len std,flow byts/s,flow pkts/s,flow iat mean,flow iat std,flow iat max,flow iat min,fwd iat tot,fwd iat mean,fwd iat std,fwd iat max,fwd iat min,bwd iat tot,bwd iat mean,bwd iat std,bwd iat max,bwd iat min,fwd psh flags,bwd psh flags,fwd urg flags,bwd urg flags,fwd header len,bwd header len,fwd pkts/s,bwd pkts/s,pkt len min,pkt len max,pkt len mean,pkt len std,pkt len var,fin flag cnt,syn flag cnt,rst flag cnt,psh flag cnt,ack flag cnt,urg flag cnt,cwe flag count,ece flag cnt,down/up ratio,pkt size avg,fwd seg size avg,bwd seg size avg,fwd byts/b avg,fwd pkts/b avg,fwd blk rate avg,bwd byts/b avg,bwd pkts/b avg,bwd blk rate avg,subflow fwd pkts,subflow fwd byts,subflow bwd pkts,subflow bwd byts,init fwd win byts,init bwd win byts,fwd act data pkts,fwd seg size min,active mean,active std,active max,active min,idle mean,idle std,idle max,idle min,label,timestamp_dayofweek,timestamp_dayofweek_int,timestamp_date
32,192.168.10.3-192.168.10.5-389-49171-6,192.168.10.3,389,192.168.10.5,49171,6,2017-07-07 07:00:52,35,2,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,114285.714286,11.666667,7.767453,18.0,3.0,14.0,14.0,0.0,14.0,14.0,35.0,35.0,0.000000,35.0,35.0,0,0,0,0,40,40,57142.857143,57142.857143,0.0,0.0,0.0,0.0,0.0,0,0,1,0,1,0,0,0,1.0,0.0,0.0,0.0,0,0,0,0,0,0,2,0,2,0,-1,256,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No Label,Friday,4,2017-07-07
43,192.168.10.3-192.168.10.9-135-1028-6,192.168.10.9,1028,192.168.10.3,135,6,2017-07-07 07:01:01,45,0,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,88888.888889,15.000000,25.119713,44.0,0.0,0.0,0.0,0.0,0.0,0.0,45.0,15.0,25.119713,44.0,0.0,0,0,0,0,0,80,0.000000,88888.888889,0.0,0.0,0.0,0.0,0.0,1,0,0,0,1,0,0,0,0.0,0.0,0.0,0.0,0,0,0,0,0,0,0,0,4,0,-1,2052,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,No Label,Friday,4,2017-07-07


In [13]:
# Batch-processing: function to run the same pipeline on all CSVs

def clean_pipeline(path_in, path_out, verbose=True):
    df = safe_read_csv(path_in)
    if verbose: print('\nProcessing', path_in.name, 'original shape', df.shape)

    # 1. Dedupe
    before = len(df)
    df = df.drop_duplicates()
    if verbose: print('  deduped ->', len(df), 'removed', before-len(df))

    # 2. Missing/infinite
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    for col in numeric_cols:
        if df[col].isnull().any():
            if any(x in col.lower() for x in ['packet', 'pkt', 'byte', 'len']):
                df[col] = df[col].fillna(0)
            else:
                df[col] = df[col].fillna(df[col].median())
    cat_cols = df.select_dtypes(include=['object']).columns.tolist()
    for col in cat_cols:
        if df[col].isnull().any():
            try:
                df[col] = df[col].fillna(df[col].mode()[0])
            except Exception:
                df[col] = df[col].fillna('')

    # 3. Timestamp parsing
    tcols = [c for c in df.columns if 'time' in c.lower() or 'timestamp' in c.lower()]
    for c in tcols:
        try:
            df[c] = pd.to_datetime(df[c])
            df[f'{c}_dayofweek'] = df[c].dt.day_name()
            df[f'{c}_dayofweek_int'] = df[c].dt.dayofweek
            df[f'{c}_date'] = df[c].dt.date
        except Exception:
            pass

    # 4. Normalize names, strings
    df.columns = [c.lower().strip() for c in df.columns]
    for col in df.select_dtypes(include=['object']).columns:
        df[col] = df[col].astype(str).str.strip().str.lower()

    # 5. Ports -> Int64
    for col in [c for c in df.columns if 'port' in c]:
        df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')

    # 6. Filter broadcast/multicast/zero-pkt (safe application)
    pkt_cols = [c for c in df.columns if any(k in c for k in ['packet','pkt','len','size','byte','byts'])]
    combined_mask = pd.Series(False, index=df.index)
    for ip_col in [c for c in df.columns if 'dst' in c and 'ip' in c]:
        s = df[ip_col].fillna('').astype(str).str.strip()
        combined_mask = combined_mask | (s == '255.255.255.255')
        first_oct = s.str.split('.', expand=True)[0]
        combined_mask = combined_mask | first_oct.map(lambda x: str(x).isdigit() and 224 <= int(x) <= 239)
    mask_zero = pd.Series(False, index=df.index)
    for c in pkt_cols:
        pktnum = pd.to_numeric(df[c], errors='coerce')
        mask_zero = mask_zero | pktnum.isna() | (pktnum == 0)
    combined_mask = combined_mask | mask_zero
    if combined_mask.sum() < len(df):
        df = df.loc[~combined_mask].copy()
    else:
        # If it would remove all rows, skip
        pass

    # 7. Outlier removal (IQR sequentially)
    numeric_cols = [c for c in df.select_dtypes(include=[np.number]).columns if 'port' not in c]
    for c in numeric_cols:
        Q1 = df[c].quantile(0.25)
        Q3 = df[c].quantile(0.75)
        IQR = Q3 - Q1
        if pd.isna(IQR) or IQR == 0:
            continue
        lower, upper = Q1 - 3*IQR, Q3 + 3*IQR
        df = df[(df[c] >= lower) & (df[c] <= upper)]

    # final save
    save_csv(df, path_out)
    return df

# Run over all csv files found earlier
summary = []
for p in csv_files:
    out = output_dir / f'cleaned_{p.name}'
    try:
        cleaned = clean_pipeline(p, out, verbose=True)
        summary.append((p.name, cleaned.shape[0], cleaned.shape[1]))
    except Exception as e:
        print('Error cleaning', p.name, e)

print('\nBatch processing finished. Summary:')
for s in summary:
    print(' -', s[0], '-> rows:', s[1], 'cols:', s[2])



Processing Friday-WorkingHours.pcap_Flow.csv original shape (703283, 84)
  deduped -> 703278 removed 5
Saved: /Users/divyanshioberoi/Desktop/IIT/Fall 2025/CS597/RealTime-Network-Traffic-Classifier/Cleaned Data/cleaned_Friday-WorkingHours.pcap_Flow.csv  (shape=(185713, 87))

Processing Monday-WorkingHours.pcap_Flow.csv original shape (529918, 84)
  deduped -> 529884 removed 34
Saved: /Users/divyanshioberoi/Desktop/IIT/Fall 2025/CS597/RealTime-Network-Traffic-Classifier/Cleaned Data/cleaned_Monday-WorkingHours.pcap_Flow.csv  (shape=(161971, 87))

Processing Thursday-WorkingHours.pcap_Flow.csv original shape (458967, 84)
  deduped -> 458868 removed 99
Saved: /Users/divyanshioberoi/Desktop/IIT/Fall 2025/CS597/RealTime-Network-Traffic-Classifier/Cleaned Data/cleaned_Thursday-WorkingHours.pcap_Flow.csv  (shape=(110826, 87))

Processing Tuesday-WorkingHours.pcap_Flow.csv original shape (445908, 84)
  deduped -> 445904 removed 4
Saved: /Users/divyanshioberoi/Desktop/IIT/Fall 2025/CS597/RealTi

In [14]:
# Final: generate a JSON schema summary for cleaned files (optional)
import json
schema = {}
for p in sorted(output_dir.glob('cleaned_*.csv')):
    try:
        d = pd.read_csv(p, nrows=200, low_memory=False, parse_dates=[c for c in pd.read_csv(p, nrows=1).columns if 'time' in c.lower() or 'timestamp' in c.lower()])
        schema[p.name] = { 'columns': {c: str(dtype) for c, dtype in d.dtypes.items()}, 'rows_sample': len(d)}
    except Exception:
        try:
            d = pd.read_csv(p, nrows=10, low_memory=False)
            schema[p.name] = { 'columns': {c: str(dtype) for c, dtype in d.dtypes.items()}, 'rows_sample': len(d)}
        except Exception as e:
            schema[p.name] = {'error': str(e)}

out_schema = output_dir / 'cleaned_schema.json'
with open(out_schema, 'w') as f:
    json.dump(schema, f, indent=2)
print('Wrote schema to', out_schema)


Wrote schema to /Users/divyanshioberoi/Desktop/IIT/Fall 2025/CS597/RealTime-Network-Traffic-Classifier/Cleaned Data/cleaned_schema.json
